# Évaluation post-labellisation — Batch 01

**Pré-requis :** l'audiologiste a rempli la colonne `label` dans `labels/batch_01_to_label.csv`  
- `0` = normal  
- `1` = anomalie  
- `2` = incertain (exclu de l'évaluation)

Ce notebook :
1. Sépare test set / pool d'entraînement
2. Compare règle, ML et consensus sur le pool d'entraînement
3. Entraîne un Random Forest supervisé
4. Compare toutes les stratégies
5. Exporte les 50 cas les plus incertains pour le tour 2 (active learning)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    precision_recall_curve, classification_report, confusion_matrix
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import joblib

from src.features import build_feature_matrix, STANDARD_FREQS

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

LABELS_PATH  = Path('../labels/batch_01_to_label.csv')
SCORES_PATH  = Path('../results/abs/scores.csv')
DATA_PATH    = Path('../data/clean_dataset.json')
OUTPUT_DIR   = Path('../labels')

VISIT_LABEL  = {0: 'Baseline', 1: 'Periodic', 2: 'Depart'}
GENDER_LABEL = {1: 'Femme', 2: 'Homme'}

## 1. Chargement des données

In [ ]:
labels_df = pd.read_csv(LABELS_PATH, index_col=0)

# Vérifier que la colonne label est bien remplie
n_vide = labels_df['label'].isna().sum() + (labels_df['label'] == '').sum()
if n_vide > 0:
    print(f"⚠  {n_vide} cases sans label — vérifier que l'audiologiste a bien rempli le fichier")
else:
    print(f"✓  {len(labels_df)} labels chargés")

labels_df['label'] = pd.to_numeric(labels_df['label'], errors='coerce')

# Exclure les incertains (label=2) de l'évaluation
n_incertain = (labels_df['label'] == 2).sum()
labels_clean = labels_df[labels_df['label'].isin([0, 1])].copy()
print(f"  Cas exploitables : {len(labels_clean)} ({n_incertain} incertains exclus)")
print(f"  Répartition : {labels_clean['label'].value_counts().to_dict()}")
print(f"\nRépartition par groupe de labellisation :")
print(labels_clean.groupby('labeling_group')['label'].value_counts().unstack(fill_value=0).to_string())

In [ ]:
# Charger scores ML et features
scores_df = pd.read_csv(SCORES_PATH, index_col=0)

df = pd.read_json(DATA_PATH, orient='records')
for col in ('dots_left', 'dots_right'):
    df[col] = df[col].apply(lambda d: {float(k): v for k, v in d.items()} if isinstance(d, dict) else {})
df['visit_date'] = pd.to_datetime(df['visit_date'], utc=True, errors='coerce')

feature_df, _ = build_feature_matrix(df)
scores_df.index  = df.index
feature_df.index = df.index

# Aligner les labels avec les scores
labeled_idx = labels_clean.index
scores_labeled   = scores_df.loc[labeled_idx]
features_labeled = feature_df.loc[labeled_idx]
y_true           = labels_clean['label'].astype(int)

print(f"Features shape : {features_labeled.shape}")
print(f"Scores shape   : {scores_labeled.shape}")

## 2. Séparation test set / pool d'entraînement

**Les 30 cas du test set sont mis de côté et ne seront plus touchés jusqu'à l'évaluation finale.**

In [ ]:
np.random.seed(42)

# Stratification : ~6 cas par groupe de labellisation
n_per_group = 6
test_idx = []
for group, grp_df in labels_clean.groupby('labeling_group'):
    n = min(n_per_group, len(grp_df))
    # Équilibrer anomalie/normal dans le test set si possible
    pos = grp_df[grp_df['label'] == 1]
    neg = grp_df[grp_df['label'] == 0]
    n_pos = min(n // 2, len(pos))
    n_neg = min(n - n_pos, len(neg))
    test_idx.extend(pos.sample(n_pos, random_state=42).index.tolist())
    test_idx.extend(neg.sample(n_neg, random_state=42).index.tolist())

test_idx  = list(set(test_idx))
train_idx = [i for i in labeled_idx if i not in test_idx]

print(f"Test set  : {len(test_idx)} cas  (TIROIR — ne plus toucher)")
print(f"Train set : {len(train_idx)} cas")
print(f"\nRépartition test set :")
print(labels_clean.loc[test_idx, 'label'].value_counts().rename({0: 'Normal', 1: 'Anomalie'}).to_string())

# Sauvegarder le test set immédiatement
labels_clean.loc[test_idx].to_csv(OUTPUT_DIR / 'test_set.csv')
labels_clean.loc[train_idx].to_csv(OUTPUT_DIR / 'train_set.csv')
print(f"\n✓  test_set.csv et train_set.csv sauvegardés")

# Variables de travail
X_train   = features_labeled.loc[train_idx].values
y_train   = y_true.loc[train_idx]
s_train   = scores_labeled.loc[train_idx]

## 3. Diagnostic — Précision / Rappel / F1 des stratégies existantes

Évaluation sur le **pool d'entraînement** (100 cas).

In [ ]:
def metrics(y_true, y_pred, name):
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f = f1_score(y_true, y_pred, zero_division=0)
    n_flagged = int(y_pred.sum())
    return {'Stratégie': name, 'Précision': round(p, 3), 'Rappel': round(r, 3),
            'F1': round(f, 3), 'N flaggés': n_flagged}

results = []

# Règle NIHL seule
results.append(metrics(y_train, s_train['nihl_flag'].fillna(0).astype(int), 'Règle NIHL'))

# Règle Ménière seule
if 'meniere_flag' in s_train.columns:
    results.append(metrics(y_train, s_train['meniere_flag'].fillna(0).astype(int), 'Règle Ménière'))

# Règle combinée (NIHL OU Ménière)
rule_combined = ((s_train['nihl_flag'].fillna(0) == 1) | (s_train['meniere_flag'].fillna(0) == 1)).astype(int)
results.append(metrics(y_train, rule_combined, 'Règle combinée (NIHL|Ménière)'))

# ML consensus (2/3)
results.append(metrics(y_train, s_train['anomaly_consensus'].fillna(0).astype(int), 'ML consensus (2/3)'))

# Consensus fort : règle ET ML
consensus_fort = ((rule_combined == 1) & (s_train['anomaly_consensus'].fillna(0) == 1)).astype(int)
results.append(metrics(y_train, consensus_fort, 'Consensus fort (règle ET ML)'))

results_df = pd.DataFrame(results).set_index('Stratégie')
print(results_df.to_string())

# Visualisation
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(results_df))
w = 0.25
ax.bar(x - w, results_df['Précision'], w, label='Précision', color='steelblue', alpha=0.85)
ax.bar(x,     results_df['Rappel'],    w, label='Rappel',    color='darkorange', alpha=0.85)
ax.bar(x + w, results_df['F1'],        w, label='F1',        color='mediumseagreen', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(results_df.index, rotation=20, ha='right')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Comparaison des stratégies — pool entraînement')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Courbe Précision / Rappel — Autoencoder

Montre quel seuil de reconstruction_error est optimal.

In [ ]:
ae_score = s_train['reconstruction_error'].values

precision_vals, recall_vals, thresholds = precision_recall_curve(y_train, ae_score)
f1_vals = np.where(
    (precision_vals + recall_vals) > 0,
    2 * precision_vals * recall_vals / (precision_vals + recall_vals),
    0
)
best_idx = np.argmax(f1_vals)
best_thr = thresholds[best_idx] if best_idx < len(thresholds) else thresholds[-1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Courbe PR
axes[0].plot(recall_vals, precision_vals, color='steelblue', linewidth=2)
axes[0].scatter(recall_vals[best_idx], precision_vals[best_idx],
                color='red', zorder=5, s=80, label=f'Seuil optimal F1 = {f1_vals[best_idx]:.2f}')
axes[0].set_xlabel('Rappel')
axes[0].set_ylabel('Précision')
axes[0].set_title('Courbe Précision/Rappel — Autoencoder')
axes[0].legend()
axes[0].grid(alpha=0.3)

# F1 en fonction du seuil
axes[1].plot(thresholds, f1_vals[:-1], color='mediumseagreen', linewidth=2)
axes[1].axvline(best_thr, color='red', linestyle='--', linewidth=1.5,
                label=f'Seuil optimal = {best_thr:.3f}')
axes[1].axvline(np.percentile(ae_score, 95), color='gray', linestyle=':', linewidth=1.5,
                label=f'P95 actuel = {np.percentile(ae_score, 95):.3f}')
axes[1].set_xlabel('Seuil reconstruction_error')
axes[1].set_ylabel('F1')
axes[1].set_title('F1 selon le seuil Autoencoder')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Seuil optimal Autoencoder : {best_thr:.4f}")
print(f"  Précision : {precision_vals[best_idx]:.3f}")
print(f"  Rappel    : {recall_vals[best_idx]:.3f}")
print(f"  F1        : {f1_vals[best_idx]:.3f}")
print(f"\nSeuil actuel P95 : {np.percentile(ae_score, 95):.4f}")

## 5. Entraînement — Random Forest supervisé

In [ ]:
# Imputer les NaN (même logique que le pipeline principal)
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)

clf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',  # compense le déséquilibre normal/anomalie
    max_depth=6,              # évite l'overfitting sur petit dataset
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1,
)

# Cross-validation stratifiée pour évaluer sans biais sur le train set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = cross_val_predict(clf, X_train_imp, y_train, cv=cv, method='predict')
y_proba_cv = cross_val_predict(clf, X_train_imp, y_train, cv=cv, method='predict_proba')[:, 1]

print("Random Forest — Cross-validation 5-fold (train set) :")
print(classification_report(y_train, y_pred_cv, target_names=['Normal', 'Anomalie']))

# Entraîner le modèle final sur tout le train set
clf.fit(X_train_imp, y_train)
joblib.dump(clf, OUTPUT_DIR / 'rf_v1.joblib')
joblib.dump(imputer, OUTPUT_DIR / 'imputer_rf_v1.joblib')
print("\n✓  Modèle sauvegardé → labels/rf_v1.joblib")

In [ ]:
# Ajouter RF aux résultats
rf_metrics = metrics(y_train, y_pred_cv, 'Random Forest (CV 5-fold)')
results.append(rf_metrics)

# Courbe PR du RF
p_rf, r_rf, thr_rf = precision_recall_curve(y_train, y_proba_cv)
f1_rf = np.where((p_rf + r_rf) > 0, 2 * p_rf * r_rf / (p_rf + r_rf), 0)
best_rf = np.argmax(f1_rf)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(r_rf, p_rf, color='mediumpurple', linewidth=2, label='Random Forest')
ax.plot(recall_vals, precision_vals, color='steelblue', linewidth=2, linestyle='--', label='Autoencoder')
ax.scatter(r_rf[best_rf], p_rf[best_rf], color='mediumpurple', s=80, zorder=5)
ax.scatter(recall_vals[best_idx], precision_vals[best_idx], color='steelblue', s=80, zorder=5)
ax.set_xlabel('Rappel')
ax.set_ylabel('Précision')
ax.set_title('Courbes PR — RF vs Autoencoder')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Comparaison finale de toutes les stratégies

In [ ]:
results_df = pd.DataFrame(results).set_index('Stratégie')
print(results_df.sort_values('F1', ascending=False).to_string())

# Matrice de confusion du meilleur modèle
best_strat_name = results_df['F1'].idxmax()
print(f"\nMeilleure stratégie : {best_strat_name}")

if best_strat_name == 'Random Forest (CV 5-fold)':
    cm = confusion_matrix(y_train, y_pred_cv)
else:
    print("(Matrice de confusion disponible pour le Random Forest uniquement)")
    cm = confusion_matrix(y_train, y_pred_cv)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Prédit Normal', 'Prédit Anomalie'],
            yticklabels=['Vrai Normal', 'Vraie Anomalie'],
            linewidths=0.5, cbar=False)
ax.set_title('Matrice de confusion — Random Forest (CV)')
plt.tight_layout()
plt.show()

## 7. Importance des features

In [ ]:
feat_names = features_labeled.columns.tolist()
importances = pd.Series(clf.feature_importances_, index=feat_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
importances.head(20).plot(kind='bar', ax=ax, color='steelblue', edgecolor='white', alpha=0.85)
ax.set_ylabel('Importance')
ax.set_title('Top 20 features — Random Forest')
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("Top 10 features :")
print(importances.head(10).round(4).to_string())

## 8. Active learning — Export des 50 cas incertains (Tour 2)

On applique le modèle sur les cas **non encore labellisés** et on extrait les 50 plus incertains.

In [ ]:
# Indices des cas déjà labellisés (train + test)
already_labeled = set(labeled_idx.tolist())

# Cas non labellisés
unlabeled_idx = [i for i in df.index if i not in already_labeled]
X_unlabeled   = imputer.transform(feature_df.loc[unlabeled_idx].values)

# Probabilité d'anomalie selon le RF
proba_anomalie = clf.predict_proba(X_unlabeled)[:, 1]

# Incertitude = proximité à 0.5
incertitude = np.abs(proba_anomalie - 0.5)

uncertainty_df = pd.DataFrame({
    'proba_anomalie': proba_anomalie,
    'incertitude':    incertitude,
}, index=unlabeled_idx)

# Top 50 cas les plus incertains
top50_idx = uncertainty_df['incertitude'].nsmallest(50).index

export_tour2 = scores_df.loc[top50_idx, [
    c for c in ['reconstruction_error', 'anomaly_consensus', 'nihl_flag', 'meniere_flag']
    if c in scores_df.columns
]].copy()
export_tour2['proba_anomalie_rf'] = uncertainty_df.loc[top50_idx, 'proba_anomalie'].round(3)
export_tour2['patient']           = df.loc[top50_idx, 'patient'].values
export_tour2['visit_date']        = df.loc[top50_idx, 'visit_date'].dt.strftime('%Y-%m-%d').values
export_tour2['visit_category']    = df.loc[top50_idx, 'visit_category'].map(VISIT_LABEL).values
export_tour2['age_at_visit']      = df.loc[top50_idx, 'age_at_visit'].round(1).values
export_tour2['gender']            = df.loc[top50_idx, 'gender'].map(GENDER_LABEL).values
export_tour2['label']             = ''
export_tour2['confidence']        = ''
export_tour2['comment']           = ''

out_path = OUTPUT_DIR / 'batch_02_to_label.csv'
export_tour2.to_csv(out_path)
print(f"✓  {len(export_tour2)} cas exportés → {out_path}")

# Distribution des probabilités du tour 2
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(proba_anomalie, bins=40, color='lightgray', edgecolor='white', alpha=0.8, label='Tous non-labellisés')
ax.hist(uncertainty_df.loc[top50_idx, 'proba_anomalie'], bins=20,
        color='orange', edgecolor='white', alpha=0.9, label='Top 50 incertains')
ax.axvline(0.5, color='red', linestyle='--', linewidth=1.5, label='Seuil 0.5')
ax.set_xlabel('Probabilité anomalie (Random Forest)')
ax.set_ylabel('Nombre de cas')
ax.set_title('Distribution des probabilités — sélection tour 2')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Résumé

| Élément | Valeur |
|---------|--------|
| Labels utilisés (train) | — |
| Test set (tiroir) | 30 cas → `labels/test_set.csv` |
| Modèle v1 | `labels/rf_v1.joblib` |
| Tour 2 à labelliser | 50 cas → `labels/batch_02_to_label.csv` |

**Prochaine étape :** envoyer `batch_02_to_label.csv` à l'audiologiste, puis relancer ce notebook avec les nouveaux labels fusionnés.